<a href="https://colab.research.google.com/github/HectorArielBaez/RegresionAvanzada/blob/main/Comparado_RL_RF_XGBOOST_svm_epilepcia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo regresion logistica usando

In [ ]:
from google.colab import files
files.upload()  # selecciona kaggle.json desde tu PC


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"arielbaez","key":"bd5916039ad5e50e7ae33fd4af856b10"}'}

In [ ]:
# 1. Instalar Kaggle y subir la API token
!pip install kaggle

# Luego, subí tu archivo `kaggle.json` en el directorio raíz de Colab

# 2. Descargar dataset de Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Por ejemplo, con el dataset 'harunshimanto/epileptic-seizure-recognition'
!kaggle datasets download -d harunshimanto/epileptic-seizure-recognition

# 3. Descomprimir
!unzip -o epileptic-seizure-recognition.zip


In [ ]:
# ==========================================
# 2. Importar librerías
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC


In [ ]:
# 5. Cargar dataset CSV descargado
df = pd.read_csv("Epileptic Seizure Recognition.csv")
# Mostrar las primeras filas
print("Dimensiones:", df.shape)
print(df.head())


Dimensiones: (11500, 180)
      Unnamed   X1   X2   X3   X4   X5   X6   X7   X8   X9  ...  X170  X171  \
0  X21.V1.791  135  190  229  223  192  125   55   -9  -33  ...   -17   -15   
1  X15.V1.924  386  382  356  331  320  315  307  272  244  ...   164   150   
2     X8.V1.1  -32  -39  -47  -37  -32  -36  -57  -73  -85  ...    57    64   
3   X16.V1.60 -105 -101  -96  -92  -89  -95 -102 -100  -87  ...   -82   -81   
4   X20.V1.54   -9  -65  -98 -102  -78  -48  -16    0  -21  ...     4     2   

   X172  X173  X174  X175  X176  X177  X178  y  
0   -31   -77  -103  -127  -116   -83   -51  4  
1   146   152   157   156   154   143   129  1  
2    48    19   -12   -30   -35   -35   -36  5  
3   -80   -77   -85   -77   -72   -69   -65  5  
4   -12   -32   -41   -65   -83   -89   -73  5  

[5 rows x 180 columns]


In [ ]:
# 6. Preparar etiquetas binarias (1 = epilepsia, 0 = no epilepsia)
df["y"] = df["y"].apply(lambda x: 1 if x == 1 else 0)
X = df.drop(columns=["y", "Unnamed"])
y = df["y"]

In [ ]:
# ==========================================
# 5. Separar train/test
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

In [ ]:
# 8. Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==========================================
# 7. Modelo Regresión Logística
# ==========================================
log_reg = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

In [ ]:
# ==========================================
# 8. Predicciones y evaluación
# ==========================================
y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))


Accuracy: 0.7086956521739131
ROC-AUC: 0.5230324511657215

Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.85      0.78      0.81      2760
           1       0.33      0.43      0.37       690

    accuracy                           0.71      3450
   macro avg       0.59      0.60      0.59      3450
weighted avg       0.74      0.71      0.72      3450



In [ ]:
# ==========================================
# 9. Modelo SVM
# ==========================================
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)
y_proba_svm = svm_model.predict_proba(X_test_scaled)[:, 1]
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_roc_auc = roc_auc_score(y_test, y_proba_svm)
svm_classification_report = classification_report(y_test, y_pred_svm, output_dict=True)

print("SVM Accuracy:", svm_accuracy)
print("SVM ROC-AUC:", svm_roc_auc)
print("\nSVM Classification Report:\n", classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9704347826086956
SVM ROC-AUC: 0.9945074564167191

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98      2760
           1       0.95      0.90      0.92       690

    accuracy                           0.97      3450
   macro avg       0.96      0.94      0.95      3450
weighted avg       0.97      0.97      0.97      3450



In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))
print(classification_report(y_test, y_pred_rf))

Random Forest
Accuracy: 0.9698550724637681
ROC-AUC: 0.9951858853182106
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      2760
           1       0.97      0.88      0.92       690

    accuracy                           0.97      3450
   macro avg       0.97      0.94      0.95      3450
weighted avg       0.97      0.97      0.97      3450



In [ ]:
# --- XGBoost ---
xgb = XGBClassifier(n_estimators=200, random_state=42, scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]))
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print("\nXGBoost")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))
print(classification_report(y_test, y_pred_xgb))


XGBoost
Accuracy: 0.9747826086956521
ROC-AUC: 0.9950378071833648
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2760
           1       0.95      0.92      0.94       690

    accuracy                           0.97      3450
   macro avg       0.97      0.95      0.96      3450
weighted avg       0.97      0.97      0.97      3450



In [ ]:
import pandas as pd

# --- Evaluar Logistic Regression ---
y_pred_lr = log_reg.predict(X_test_scaled)
res_lr = {
    "Modelo": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "ROC-AUC": roc_auc_score(y_test, log_reg.predict_proba(X_test_scaled)[:,1]),
    "Precision_0": classification_report(y_test, y_pred_lr, output_dict=True)["0"]["precision"],
    "Recall_0": classification_report(y_test, y_pred_lr, output_dict=True)["0"]["recall"],
    "Precision_1": classification_report(y_test, y_pred_lr, output_dict=True)["1"]["precision"],
    "Recall_1": classification_report(y_test, y_pred_lr, output_dict=True)["1"]["recall"]
}
# Crear un diccionario con los resultados del modelo SVM
res_svm = {
    "Modelo": "SVM",
    "Accuracy": svm_accuracy,
    "ROC-AUC": svm_roc_auc,
    "Precision_0": svm_classification_report["0"]["precision"],
    "Recall_0": svm_classification_report["0"]["recall"],
    "Precision_1": svm_classification_report["1"]["precision"],
    "Recall_1": svm_classification_report["1"]["recall"]
}
# --- Evaluar Random Forest ---
y_pred_rf = rf.predict(X_test)
res_rf = {
    "Modelo": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]),
    "Precision_0": classification_report(y_test, y_pred_rf, output_dict=True)["0"]["precision"],
    "Recall_0": classification_report(y_test, y_pred_rf, output_dict=True)["0"]["recall"],
    "Precision_1": classification_report(y_test, y_pred_rf, output_dict=True)["1"]["precision"],
    "Recall_1": classification_report(y_test, y_pred_rf, output_dict=True)["1"]["recall"]
}

# --- Evaluar XGBoost ---
y_pred_xgb = xgb.predict(X_test)
res_xgb = {
    "Modelo": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred_xgb),
    "ROC-AUC": roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]),
    "Precision_0": classification_report(y_test, y_pred_xgb, output_dict=True)["0"]["precision"],
    "Recall_0": classification_report(y_test, y_pred_xgb, output_dict=True)["0"]["recall"],
    "Precision_1": classification_report(y_test, y_pred_xgb, output_dict=True)["1"]["precision"],
    "Recall_1": classification_report(y_test, y_pred_xgb, output_dict=True)["1"]["recall"]
}


In [ ]:

# Crear un DataFrame temporal a partir del diccionario res_svm
#temp_df_svm = pd.DataFrame.from_records([res_svm])
results_df = pd.DataFrame([res_lr, res_rf, res_xgb, res_svm])
# Concatenar el DataFrame temporal del modelo SVM al DataFrame results_df existente
#results_df = pd.concat([results_df, temp_df_svm], ignore_index=True)

# Mostrar la tabla actualizada
display(results_df)

,Modelo,Accuracy,ROC-AUC,Precision_0,Recall_0,Precision_1,Recall_1
0,Logistic Regression,0.708696,0.523032,0.845336,0.778261,0.326733,0.430435
1,Random Forest,0.969855,0.995186,0.970255,0.992754,0.968051,0.878261
2,XGBoost,0.974783,0.995038,0.979892,0.988768,0.953383,0.918841
3,SVM,0.970435,0.994507,0.974304,0.989130,0.953704,0.895652


In [ ]:
# Guardado de resultados
results_df.to_csv("resultados_modelos.csv", index=False)
print("La tabla ha sido exportada a 'resultados_modelos.csv'")

La tabla ha sido exportada a 'resultados_modelos.csv'


In [ ]:
# Texto de la explicación a agregar
explanation = """

Explicación de las Métricas:

Precision_0 (Precisión para la clase 0 - No Epilepsia): De todas las instancias que el modelo predijo como clase 0 (no epilepsia), ¿qué proporción realmente pertenecían a la clase 0? Una alta precisión para la clase 0 significa que cuando el modelo dice que no hay epilepsia, es muy probable que sea correcto.

Precision_1 (Precisión para la clase 1 - Epilepsia): De todas las instancias que el modelo predijo como clase 1 (epilepsia), ¿qué proporción realmente pertenecían a la clase 1? Una alta precisión para la clase 1 significa que cuando el modelo dice que hay epilepsia, es muy probable que sea correcto.

Recall_0 (Exhaustividad para la clase 0 - No Epilepsia): De todas las instancias que realmente pertenecían a la clase 0 (no epilepsia), ¿qué proporción fueron correctamente identificadas por el modelo como clase 0? Una alta exhaustividad para la clase 0 significa que el modelo es bueno para encontrar todas las instancias negativas.

Recall_1 (Exhaustividad para la clase 1 - Epilepsia): De todas las instancias que realmente pertenecían a la clase 1 (epilepsia), ¿qué proporción fueron correctamente identificadas por el modelo como clase 1? Una alta exhaustividad para la clase 1 significa que el modelo es bueno para encontrar todos los casos positivos (epilepsia).

En resumen:
Precision responde a la pregunta: "De todo lo que predije como positivo/negativo, ¿cuánto fue realmente positivo/negativo?"
Exhaustividad responde a la pregunta: "De todo lo que realmente era positivo/negativo, ¿cuánto fui capaz de identificar correctamente?"
"""

# Abrir el archivo en modo append ('a') y agregar la explicación
with open("resultados_modelos.csv", "a") as f:
    f.write(explanation)

print("La explicación ha sido agregada a 'resultados_modelos.csv'")

La explicación ha sido agregada a 'resultados_modelos.csv'
